Se requiere obtener información sobre el `total del presupuesto` y `total de ingresos` de las `películas` donde el `año de la fecha de lanzamiento` debe ser mayor o igual a 2015, también debe estar agrupado por el `año de la fecha de lanzamiento` y el `país` al que pertenece cada película.
También se requiere realizar un ranking ordenado de manera descendente por el `total del presupuesto` y `total de ingresos` particionado por el `Año de la fecha de lanzamiento`

In [0]:
%run "../includes/configuration"

In [0]:
%run "../includes/common_functions"

In [0]:
dbutils.widgets.text("p_file_date", "2024-12-16")
v_file_date = dbutils.widgets.get("p_file_date")

## 1. Obtenemos las películas y campos que nos interesan (`año de lanzamiento`, `budget`, `revenue`)

In [0]:
movies_df = spark.read.table("movie_silver.movies").filter(f"file_date = '{v_file_date}'")
movies_filtered_df = (movies_df
    .filter(movies_df.year_release_date >= 2015)
    .select("movie_id", "year_release_date", "budget", "revenue")
)
display(movies_filtered_df)


movie_id,year_release_date,budget,revenue


## 2. Obtener el `nombre del país` de la película

In [0]:
production_countries_df = spark.read.table("movie_silver.productions_countries").filter(f"file_date = '{v_file_date}'")
countries_df = spark.read.table("movie_silver.countries")

movies_countries_name_df = (
    production_countries_df.join(countries_df, on="country_id", how="inner")
    .select("movie_id", "country_name")
)
display(movies_countries_name_df)

movie_id,country_name
5,United States of America
11,United States of America
12,United States of America
13,United States of America
14,United States of America
16,Argentina
16,Denmark
16,Finland
16,France
16,Germany


## 3. Añadir `nombre del país` al DataFrame

In [0]:
movies_final_df = (
    movies_filtered_df.join(movies_countries_name_df, on="movie_id", how="inner")
    .select("year_release_date", "budget", "revenue", "country_name")
)

display(movies_final_df)

year_release_date,budget,revenue,country_name


## 4. Agrupar por el `año de la fecha de lanzamiento` y el `género`

In [0]:
from pyspark.sql.functions import sum
results_group_by_df = ( movies_final_df
                       .groupBy("year_release_date", "country_name")
                       .agg(
                           sum("budget").alias("total_budget"),
                           sum("revenue").alias("total_revenue"),
                       )
)

## 5. Crear ranking ordenado de manera ascendente por el `total del presupuesto` y `total de ingresos` particionado por el `Año de la fecha de lanzamiento`

In [0]:
from pyspark.sql.functions import dense_rank, desc, lit
from pyspark.sql.window import Window

results_dense_rank_df = Window.partitionBy("year_release_date").orderBy(desc("total_budget"), desc("total_revenue"))
results_rank_df = results_group_by_df.withColumn("rank", dense_rank().over(results_dense_rank_df)).withColumn("created_date", lit(v_file_date))


display(results_rank_df)

year_release_date,country_name,total_budget,total_revenue,rank,created_date


## 6. Escribir datos en el DataLake en formato `Delta`

In [0]:
merge_delta_lake( results_rank_df, "movie_gold", "results_group_movie_country", "tgt.year_release_date = src.year_release_date AND tgt.country_name = src.country_name AND tgt.created_date = src.created_date", "created_date" )

In [0]:
%sql
SELECT * FROM movie_gold.results_group_movie_country

year_release_date,country_name,total_budget,total_revenue,rank,created_date
2015,United States of America,5.143775E9,1.8450384353E10,1,2024-12-30
2015,United Kingdom,6.5152236E8,1.894996027E9,2,2024-12-30
2015,Canada,3.245E8,1.334394558E9,3,2024-12-30
2015,Germany,3.095E8,9.5835032E8,4,2024-12-30
2015,China,2.8E8,8.8867852E8,5,2024-12-30
2015,Australia,2.13E8,6.79034882E8,6,2024-12-30
2015,Japan,1.9E8,1.50624936E9,7,2024-12-30
2015,France,1.51E8,2.06495048E8,8,2024-12-30
2015,Hong Kong,1.35E8,5.32950503E8,9,2024-12-30
2015,Taiwan,1.35E8,5.32950503E8,9,2024-12-30
